# DF Hyperparameter Sweep & Calibration

**Two experiments**:
1. **Lambda Sweep**: EDL+ORCU — scan λ_orcu × λ_reg (9 combos, 50 epochs each)
2. **Temperature Calibration**: Load trained EDL model, apply T=0.5~5.0 scaling

**Expected runtime**: ~2h on T4 x2 (9×50ep + 1 evaluation pass)

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), \
    "Git clone failed! Check repo visibility."
print("Code cloned.")

In [ ]:
# ---- Master Setup ----
import os, sys, json, copy, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Pre-download weights
print("Downloading ViT weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("Done.")

CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"Device: {device} | GPU: {gpu_name}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
print("Setup complete.")

## 2. Evaluation Helper

In [ ]:
@torch.no_grad()
def evaluate(model, loader, temperature=1.0):
    """Full evaluation with optional temperature scaling."""
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha, dim=0) if all_alpha else None
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets, temperature=temperature)

print("evaluate() ready.")

## 3. Lambda Sweep — EDL+ORCU

Sweep λ_orcu ∈ {0.1, 0.3, 0.5} × λ_reg ∈ {0.005, 0.01, 0.02} = 9 runs, 50 epochs each.
Best combo selected by val Acc, final evaluation on test set.

In [ ]:
import os, sys, json, copy, itertools, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
from data import create_dataloaders
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Grid
lambda_orcu_vals = [0.1, 0.3, 0.5]
lambda_reg_vals = [0.005, 0.01, 0.02]
sweep_epochs = 50
sweep_seed = 42

print("="*60)
print("EDL+ORCU Lambda Sweep — DF")
print(f"  lambda_orcu: {lambda_orcu_vals}")
print(f"  lambda_reg:  {lambda_reg_vals}")
print(f"  epochs: {sweep_epochs} per run")
print(f"  total: {len(lambda_orcu_vals) * len(lambda_reg_vals)} runs")
print("="*60)

# Data (shared)
train_loader, val_loader, test_loader = create_dataloaders(
    DATA_ROOT, task="df", batch_size=32, num_workers=2)
print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")

sweep_results = []
best_val_acc = 0.0
best_combo = None
t_start = time.time()

for lam_orcu, lam_reg in itertools.product(lambda_orcu_vals, lambda_reg_vals):
    print(f"\n  [lam_orcu={lam_orcu}, lam_reg={lam_reg}] training...")
    torch.manual_seed(sweep_seed)
    
    model = build_model(task="df", mode="edl_orcu")
    model.to(device)
    
    criterion = CombinedLoss(
        num_classes=4, lambda_orcu=lam_orcu, lambda_kl=0.1, orcu_t=3.0,
        orcu_lambda_reg=lam_reg,
        stage_1_epochs=5, stage_2_epochs=15, total_epochs=sweep_epochs)
    
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb if n.startswith("backbone") else hd).append(p)
    optimizer = optim.AdamW([{"params": bb, "lr": 1e-4}, {"params": hd, "lr": 1e-3}],
                            weight_decay=0.05)
    ws = LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cs = CosineAnnealingLR(optimizer, T_max=sweep_epochs - 5)
    scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[5])
    
    best_val = 0.0
    best_state = None
    for epoch in range(sweep_epochs):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        if (epoch + 1) % 10 == 0 or epoch == sweep_epochs - 1:
            vm = evaluate(model, val_loader)
            if vm["acc"] > best_val:
                best_val = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())
    
    model.load_state_dict(best_state)
    tm = evaluate(model, test_loader)
    
    result = {
        "lambda_orcu": lam_orcu, "lambda_reg": lam_reg,
        "val_acc": best_val,
        "test_acc": tm["acc"], "test_qwk": tm["qwk"],
        "test_ece": tm["ece"], "test_unim": tm["pct_unimodal"],
        "test_u_ece": tm["u_ece"], "test_auroc_u": tm["auroc_u"],
        "test_f1": tm["macro_f1"],
    }
    sweep_results.append(result)
    
    if best_val > best_val_acc:
        best_val_acc = best_val
        best_combo = (lam_orcu, lam_reg)
    
    print(f"    Val Acc={best_val:.4f} | Test Acc={tm['acc']:.4f} QWK={tm['qwk']:.4f} "
          f"ECE={tm['ece']:.4f} Unim={tm['pct_unimodal']:.2%}")

elapsed = (time.time() - t_start) / 60

# Summary
print(f"\n{'='*80}")
print(f"Lambda Sweep Results (sorted by Val Acc) — {elapsed:.1f} min")
print(f"{'='*80}")
print(f"{'lam_orcu':>8} {'lam_reg':>8} {'Val':>8} {'Test':>8} {'QWK':>8} {'F1':>8} {'ECE':>8} {'Unim':>8} {'U-ECE':>8} {'AUROCu':>8}")
print("-" * 88)
for r in sorted(sweep_results, key=lambda x: x["val_acc"], reverse=True):
    print(f"{r['lambda_orcu']:8.3f} {r['lambda_reg']:8.4f} "
          f"{r['val_acc']:8.4f} {r['test_acc']:8.4f} {r['test_qwk']:8.4f} {r['test_f1']:8.4f} "
          f"{r['test_ece']:8.4f} {r['test_unim']:7.2%} {r['test_u_ece']:8.4f} {r['test_auroc_u']:8.4f}")

print(f"\nBest: lam_orcu={best_combo[0]}, lam_reg={best_combo[1]} (Val Acc={best_val_acc:.4f})")

# Save
with open(os.path.join(OUTPUT_DIR, "lambda_sweep_v2.json"), "w") as f:
    json.dump({"results": sweep_results, "best_combo": list(best_combo)}, f, indent=2)
print("Saved to lambda_sweep_v2.json")

## 4. Temperature Calibration — EDL

Load the best DF EDL model (trained in multi-seed or main experiment) and sweep temperature T.
If no pre-trained model exists, train EDL first (100 epochs).

In [ ]:
import os, sys, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
from data import create_dataloaders
from models import build_model
from eval import compute_metrics

OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 1: Try loading pre-trained EDL model
ckpt_paths = [
    os.path.join(OUTPUT_DIR, "df_edl_seed42", "best.pt"),
    os.path.join(OUTPUT_DIR, "df_edl", "best.pt"),
]

model = build_model(task="df", mode="edl")
loaded = False
for ckpt_path in ckpt_paths:
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded EDL checkpoint from {ckpt_path}")
        loaded = True
        break

if not loaded:
    print("No pre-trained EDL model found. Training EDL from scratch (100 epochs)...")
    model.to(device)
    
    from losses.edl_loss import EDLLoss
    train_loader, val_loader, test_loader = create_dataloaders(
        DATA_ROOT, task="df", batch_size=32, num_workers=2)
    
    criterion = EDLLoss(num_classes=4, kl_lambda=0.1)
    class EDLWrap(nn.Module):
        def forward(self, alpha, z, targets, epoch=None):
            loss = criterion(alpha, targets, epoch, 100)
            return loss, {"stage": 0, "L_edl": loss.item()}
    criterion = EDLWrap()
    
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb if n.startswith("backbone") else hd).append(p)
    optimizer = optim.AdamW([{"params": bb, "lr": 1e-4}, {"params": hd, "lr": 1e-3}],
                            weight_decay=0.05)
    ws = LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cs = CosineAnnealingLR(optimizer, T_max=95)
    scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[5])
    
    best_val_acc, best_state = 0.0, None
    for epoch in range(100):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            vm = evaluate(model, val_loader)
            if vm["acc"] > best_val_acc:
                best_val_acc = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())
    
    model.load_state_dict(best_state)
    os.makedirs(os.path.join(OUTPUT_DIR, "df_edl"), exist_ok=True)
    torch.save({"model_state_dict": best_state},
               os.path.join(OUTPUT_DIR, "df_edl", "best.pt"))
    print(f"EDL trained. Best val Acc={best_val_acc:.4f}")
else:
    _, _, test_loader = create_dataloaders(
        DATA_ROOT, task="df", batch_size=32, num_workers=2)

model.to(device)
model.eval()

# Step 2: Temperature sweep
temperatures = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]
print(f"\n{'='*60}")
print("Temperature Calibration Sweep — DF EDL")
print(f"{'='*60}")
print(f"{'T':>6} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
print("-" * 72)

temp_results = {}
for T in temperatures:
    m = evaluate(model, test_loader, temperature=T)
    temp_results[f"T_{T}"] = {
        "acc": m["acc"], "macro_f1": m["macro_f1"], "qwk": m["qwk"],
        "ece": m["ece"], "u_ece": m["u_ece"], "auroc_u": m["auroc_u"],
        "pct_unimodal": m["pct_unimodal"]
    }
    print(f"{T:6.1f} {m['acc']:8.4f} {m['macro_f1']:8.4f} {m['qwk']:8.4f} "
          f"{m['ece']:8.4f} {m['u_ece']:8.4f} {m['auroc_u']:8.4f} {m['pct_unimodal']:7.2%}")

# Find best
best_ece = min(temp_results.items(), key=lambda x: x[1]["ece"])
best_auroc = max(temp_results.items(), key=lambda x: x[1]["auroc_u"])
print(f"\nBest ECE: {best_ece[0]} -> ECE={best_ece[1]['ece']:.4f}")
print(f"Best AUROC(u): {best_auroc[0]} -> AUROC(u)={best_auroc[1]['auroc_u']:.4f}")

# Save
with open(os.path.join(OUTPUT_DIR, "temperature_sweep.json"), "w") as f:
    json.dump(temp_results, f, indent=2)
print("Saved to temperature_sweep.json")

## 5. Results Summary

Print both sweep results for easy copy-paste.

In [ ]:
import json, os

OUTPUT_DIR = "/kaggle/working"

print("="*80)
print("FINAL RESULTS")
print("="*80)

# Lambda sweep
fpath = os.path.join(OUTPUT_DIR, "lambda_sweep_v2.json")
if os.path.exists(fpath):
    with open(fpath) as f:
        ls = json.load(f)
    print(f"\n--- Lambda Sweep ---")
    print(f"Best combo: lam_orcu={ls['best_combo'][0]}, lam_reg={ls['best_combo'][1]}")
    print(f"{'lam_orcu':>8} {'lam_reg':>8} {'Val':>8} {'Test':>8} {'QWK':>8} {'F1':>8} {'ECE':>8} {'Unim':>8} {'U-ECE':>8} {'AUROC(u)':>8}")
    print("-" * 88)
    for r in sorted(ls["results"], key=lambda x: x["val_acc"], reverse=True):
        print(f"{r['lambda_orcu']:8.3f} {r['lambda_reg']:8.4f} "
              f"{r['val_acc']:8.4f} {r['test_acc']:8.4f} {r['test_qwk']:8.4f} "
              f"{r['test_f1']:8.4f} {r['test_ece']:8.4f} {r['test_unim']:7.2%} "
              f"{r['test_u_ece']:8.4f} {r['test_auroc_u']:8.4f}")
else:
    print("lambda_sweep_v2.json not found")

# Temperature sweep
fpath = os.path.join(OUTPUT_DIR, "temperature_sweep.json")
if os.path.exists(fpath):
    with open(fpath) as f:
        ts = json.load(f)
    print(f"\n--- Temperature Sweep ---")
    print(f"{'T':>6} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
    print("-" * 72)
    for T_key in sorted(ts.keys(), key=lambda k: float(k.replace("T_", ""))):
        r = ts[T_key]
        t_val = float(T_key.replace("T_", ""))
        print(f"{t_val:6.1f} {r['acc']:8.4f} {r['macro_f1']:8.4f} {r['qwk']:8.4f} "
              f"{r['ece']:8.4f} {r['u_ece']:8.4f} {r['auroc_u']:8.4f} {r['pct_unimodal']:7.2%}")
else:
    print("temperature_sweep.json not found")

print("\n=== DONE ===")
print("Copy the tables above and paste to Claude.")
print("Also download /kaggle/working/lambda_sweep_v2.json and temperature_sweep.json for backup.")